# CNN-based Keyword Spotting

Questo notebook effettua il pre-processing delle registrazioni contenute nel _Google Speech Command dataset_:
- Generazione del file di metadati
- Generazione degli spettrogrammi in scala MEL

(versione COLAB)

Il dataset raw (1.4Gb) può essere scaricato da questo [link](http://download.tensorflow.org/data/speech_commands_v0.01.tar.gz).

Per eseguire le funzioni `parse_dataset()` e `generate_dataset()`, il dataset deve essere posizionato nella directory specificata da `base_path/data_path`
<br><br><br>
Carlo Aironi<br>
c.aironi@staff.univpm.it<br>

## Download full dataset

In [ ]:
!wget http://download.tensorflow.org/data/speech_commands_v0.01.tar.gz

--2026-08-18 20:36:19--  http://download.tensorflow.org/data/speech_commands_v0.01.tar.gz
Resolving download.tensorflow.org (download.tensorflow.org)... 74.125.135.207, 173.194.43.207, 192.179.26.207, ...
Connecting to download.tensorflow.org (download.tensorflow.org)|74.125.135.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1489096277 (1.4G) [application/gzip]
Saving to: ‘speech_commands_v0.01.tar.gz’

speech_commands_v0. 100%[===================>]   1.39G   108MB/s    in 12s     

2026-08-18 20:36:31 (119 MB/s) - ‘speech_commands_v0.01.tar.gz’ saved [1489096277/1489096277]



In [ ]:
!mkdir -p /content/GSC
!tar -xzf /content/speech_commands_v0.01.tar.gz -C /content/GSC/

## Import libraries

In [ ]:
import os
import librosa
import numpy as np
from tqdm import tqdm as tq
import pandas as pd

import matplotlib.pyplot as plt

import multiprocessing
from joblib import Parallel, delayed

## Parameters

In [ ]:
base_path = '/content/drive/MyDrive/Deep Learning/'
data_path = 'data/'

mfcc_ndarray = 'gsc110_mfcc.npy'
melsp_ndarray = 'gsc110_melspecs.npy'
meta_file = 'gsc110_meta.csv'
cat_list = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine'] # seleziono le classi di interesse

SAMPLE_RATE = 16000
EXP_SAMPLES = 16000

# split train/validation partition percentage
TRAIN_PART = 0.8
VAL_PART = 0.2

FFT_SIZE = 1024
FFT_HOP = 128
FFT_MEL_BANDS = 64
N_MFCC = 20

EXP_H = 64
EXP_W = 126

## Parse dataset and make meta file

Questa funzione analizza il contenuto delle directory del dataset compilando un file di metadati che ha la struttura di una tabella, con i seguenti campi:
- `filepath`: path relativo di ogni singolo file wav
- `category`: valore numerico che indica la classe, da 0 a 9
- `train`: flag che indica l'appartenenza del file al train-set
- `validation`: flag che indica l'appartenenza del file al validation-set
- `test`: flag che indica l'appartenenza del file al test-set.

Per questa esercitazione verrà usata una suddivisione in soli due split: train e validation. La suddivisione avviene sulla base di un vettore di indici generato casualmente, che per ogni classe contiene in percentuale `TRAIN_PART` valori 0 e `VAL_PART` valori 1.

Questo metodo di creazione delle partizioni garantisce che ogni classe, indipendentemente dal numero di elementi, contenga la percentuale voluta di elementi train e validation.

Lo stesso risultato può essere ottenuto con la funzione `torch.utils.data.random_split()`, che con opportuni parametri garantisce il bilanciamento dello split anche su dataset a classi sbilanciate (_stratified split_).

In [ ]:
def parse_dataset():

    mymeta = pd.DataFrame(columns=('filepath', 'category', 'train', 'validation', 'test'))

    print('Parse dataset and generate meta file')
    meta_index=0
    for ci, category in enumerate(cat_list):
        print(f'class {category}...')

        source_path = os.path.join(base_path,data_path,category)

        nwav = len([f for f in os.listdir(source_path) if (os.path.join(source_path, f))[-3:]=='wav']) # conta gli wav

        split_indices = np.random.choice(2, nwav, p=[TRAIN_PART, VAL_PART]) # genera il tensore con gli indici di split, in questo caso solo train e valid

        wav_index = 0

        for wavfile in tq(os.listdir(source_path)):

            if wavfile[-3:] != 'wav': continue # skip a loop if file is not a wavefile

            mymeta.loc[meta_index] = ({'filepath': os.path.join(category,wavfile),
                                      'category': ci,
                                      'train': 1 if split_indices[wav_index] == 0 else 0 ,
                                      'validation': 1 if split_indices[wav_index] == 1 else 0 ,
                                      'test': 1 if split_indices[wav_index] == 2 else 0 })

            wav_index +=1
            meta_index +=1

    mymeta.to_csv(os.path.join(base_path, meta_file), index = False, header=True)
    # mymeta.to_pickle(base_path + data_path + meta_file)

    print('done')

In [ ]:
parse_dataset()

## Check dataset

Controlla se tutti i file del dataset hanno la lunghezza e la frequenza di campionamento desiderata

In [ ]:
def check_dataset():

    mymeta = pd.read_csv(base_path + meta_file)

    print('Sample rate and length check...')
    len_list = []
    sr_list = []
    for i in range(len(mymeta)):
        full_path = base_path + data_path + mymeta.iloc[i].filepath
        clip, sr = librosa.load(full_path, sr=None)

        if sr != SAMPLE_RATE:
            print(f'Found sample rate {sr} in file {mymeta.iloc[i].filepath}')

        len_list.append(len(clip))
        sr_list.append(sr)

    print('done')

    return len_list, sr_list

In [ ]:
lens, srs = check_dataset()
plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.hist(lens)
plt.title('Length')
plt.grid()
plt.subplot(1,2,2)
plt.hist(srs)
plt.title('Sample Rate')
plt.grid()
plt.show()

## Generate CNN dataset

La funzione `generate_dataset()` calcola gli spettrogrammi per ogni file wav del dataset, e li salva su un array bidimensionale, che verrà successivamente utilizzato dalla rete CNN.

La funzione viene eseguita in modalità parallela, utilizzando tutti i processori disponibili nella macchina; questo consente di velocizzare notevolmente le operazioni di FFT.

Inizialmente ogni file viene letto ed eventualmente ricampionato alla frequenza di 16kHz e la sua lunghezza viene controllata ed adattata, con operazioni di trim o zero padding, a 16000 campioni (1 secondo).

Lo spettrogramma in scala MEL viene calcolato con la funzione apposita, della libreria __Librosa__, che richiede come parametri:
- `y`: la sequenza di campioni audio
- `sr`: la frequenza di campionamento
- `n_fft`: la dimensione della finestra su cui calcolare la FFT
- `hop_length`: lo spostamento nel tempo tra due finestre consecutive
- `power`: l'opzione per il calcolo dello spettro di ampiezza (1) o di potenza (2)
- `n_mels`: il numero di bande mel (filtri triangolari)
- `fmax`: la frequenza massima di analisi (che di default è pari a SAMPLE RATE / 2)

In [ ]:
def generate_dataset(): # multiprocess version
    CPU_COUNT = multiprocessing.cpu_count()
    mymeta = pd.read_pickle(base_path + data_path + meta_file)
    rows_logmelsp = []
    rows_mfcc = []
    print(f'Start extracting spectrograms and MFCCs using {CPU_COUNT} CPUs')

    def gen_spec(idx):

        full_path = base_path + data_path + mymeta.iloc[idx].filepath # legge il metafile e crea gli spettrogrammi con lo stesso ordine

        clip, _ = librosa.load(full_path, sr=SAMPLE_RATE)

        if len(clip) > EXP_SAMPLES: # trim
            clip = clip[0:EXP_SAMPLES]

        if len(clip) < EXP_SAMPLES: # zero pad
            clip = np.pad(clip, (0, EXP_SAMPLES - len(clip)), 'constant', constant_values=(0, 0))

        # Log-mel spectrogram
        melsp = librosa.feature.melspectrogram(y=clip,
                                                sr=SAMPLE_RATE,
                                                n_fft=FFT_SIZE,
                                                hop_length=FFT_HOP,
                                                power=2,
                                                n_mels=FFT_MEL_BANDS,
                                                fmax=SAMPLE_RATE / 2)

        logmelsp = librosa.power_to_db(melsp, ref=np.max)     # transform to log_mel_spectrogram
        logmelsp = logmelsp.astype(np.float32)                # convert data to reduce dimensions

        # MFCC, calcolati a partire dal log-mel spectrogram già ottenuto
        mfcc = librosa.feature.mfcc(S=logmelsp,
                                     sr=SAMPLE_RATE,
                                     n_mfcc=N_MFCC)
        mfcc = mfcc.astype(np.float32)

        logmelsp_flat = logmelsp.flatten()[:, np.newaxis].T   # flatten
        mfcc_flat = mfcc.flatten()[:, np.newaxis].T           # flatten

        return logmelsp_flat, mfcc_flat

    results = Parallel(n_jobs=CPU_COUNT)(delayed(gen_spec)(i) for i in tq(range(len(mymeta))))

    rows_logmelsp, rows_mfcc = zip(*results)

    rows_logmelsp = np.vstack(rows_logmelsp)
    rows_mfcc = np.vstack(rows_mfcc)

    # Save in .npy files
    np.save(base_path + melsp_ndarray, rows_logmelsp)
    np.save(base_path + mfcc_ndarray, rows_mfcc)

    print('done')

In [ ]:
generate_dataset()